In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


In [ ]:
dataset = load_dataset("ag_news")

df_train = pd.DataFrame(dataset["train"])
df_test = pd.DataFrame(dataset["test"])

def clean_text(text):
  import re
  text = text.lower()
  text = re.sub(r"http\S+", "", text)
  text = re.sub(r"[^a-z0-9\s]", "", text)
  text = re.sub(r"\s+", "", text).strip()
  return text
df_train["clean_text"] = df_train["text"].apply(clean_text)
df_test["clean_text"] = df_test["text"].apply(clean_text)

print(f"Train: {len(df_train)} | Test: {len(df_test)}")

Train: 120000 | Test: 7600


In [ ]:
train_dataset = Dataset.from_pandas(df_train[["clean_text", "label"]].reset_index(drop=True))
test_dataset  = Dataset.from_pandas(df_test[["clean_text", "label"]].reset_index(drop=True))

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset  = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization done!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

✅ Tokenization done!


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)
print(f"✅ Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded! Parameters: 66,956,548


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir="distilbert-agnews",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    logging_steps=100,
    fp16=True,        # GPU speeds this up massively
    report_to="none"
)
print("✅ Training arguments set!")

✅ Training arguments set!


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Starting training on FULL dataset with GPU...")
trainer.train()

Starting training on FULL dataset with GPU...


Step,Training Loss
100,1.388497
200,1.388581
300,1.388237
400,1.389769
500,1.387584
600,1.390682
700,1.387334
800,1.386606
900,1.388800
1000,1.388561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
100,1.388497
200,1.388581
300,1.388237
400,1.389769
500,1.387584
600,1.390682
700,1.387334
800,1.386606
900,1.388800
1000,1.388561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=11250, training_loss=1.3860287896050347, metrics={'train_runtime': 1415.185, 'train_samples_per_second': 254.384, 'train_steps_per_second': 7.949, 'total_flos': 1.192249110528e+16, 'train_loss': 1.3860287896050347, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()

print("\n📊 Final Results:")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1 Macro: {results['eval_f1_macro']:.4f}")


📊 Final Results:
Accuracy: 0.2561
F1 Macro: 0.1164


In [ ]:
trainer.save_model("distilbert-agnews-final")
tokenizer.save_pretrained("distilbert-agnews-final")

print("✅ Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [24]:
import shutil
shutil.make_archive("distilbert-agnews-final", "zip", "distilbert-agnews-final")

from google.colab import files
files.download("distilbert-agnews-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>